## Load raw data

In [2]:
import pandas as pd

df = pd.read_excel("../data/raw/BLS/MSA_M2025_dl.xlsx")

print(df.shape)

(150023, 32)


## Filter Bay Area Metros

In [3]:
bay_area_metros = [
    "San Francisco-Oakland-Fremont, CA",
    "San Jose-Sunnyvale-Santa Clara, CA",
    "Santa Rosa-Petaluma, CA",
    "Napa, CA",
    "Vallejo, CA",
    "Santa Cruz-Watsonville, CA"
]
bay_area_df = df[df["AREA_TITLE"].isin(bay_area_metros)].copy()
# Verify
bay_area_df.shape

(2824, 32)

## Filter Tech Occupations

In [15]:
tech_df = bay_area_df[
    (bay_area_df["OCC_CODE"].str.startswith("15-"))
]
tech_df.shape

(95, 32)

## Remove Major Group Rows

In [16]:
tech_df = tech_df[tech_df["O_GROUP"] == "detailed"]
tech_df["O_GROUP"].value_counts()
tech_df.shape

(89, 32)

## Remove Empty Columns

In [17]:
# In the data exploration phase, we discovered that the PCT_TOTAL and PCT_RPT columns are completely empty.
# Below is the check for those columns:
tech_df[["PCT_TOTAL","PCT_RPT"]].isnull().sum()
tech_df.drop(columns=["PCT_TOTAL", "PCT_RPT"], inplace=True)
tech_df.shape

(89, 30)

## Investigate Special Symbols

In [ ]:
numeric_cols = [
    "TOT_EMP",
    "JOBS_1000",
    "LOC_QUOTIENT",
    "H_MEAN",
    "A_MEAN",
    "H_MEDIAN",
    "A_MEDIAN"
]
for col in numeric_cols:
    print(col)
    print(
        tech_df[
            ~tech_df[col].astype(str).str.match(r"^[0-9.]+$")
        ][col].unique()
    )

TOT_EMP
['**']
JOBS_1000
['**']
LOC_QUOTIENT
['**']
H_MEAN
['*']
A_MEAN
['*']
H_MEDIAN
['*']
A_MEDIAN
['*']


## Cleaning

In [19]:
# we already found symbols '#' and '*' in the numeric columns of the data, so we will remove them from the 
# dataset
tech_df = tech_df.replace(
    ["*", "#"],
    pd.NA
)

## Convert to Numeric, i.e. the columns we want them to be numeric

In [22]:
for col in numeric_cols:
    tech_df[col] = pd.to_numeric(
        tech_df[col],
        errors="coerce"
    )

In [ ]:
# verify results after convering to numeric
tech_df.info()

<class 'pandas.DataFrame'>
Index: 89 entries, 91667 to 140029
Data columns (total 30 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   AREA          89 non-null     int64  
 1   AREA_TITLE    89 non-null     str    
 2   AREA_TYPE     89 non-null     int64  
 3   PRIM_STATE    89 non-null     str    
 4   NAICS         89 non-null     int64  
 5   NAICS_TITLE   89 non-null     str    
 6   I_GROUP       89 non-null     str    
 7   OWN_CODE      89 non-null     int64  
 8   OCC_CODE      89 non-null     str    
 9   OCC_TITLE     89 non-null     str    
 10  O_GROUP       89 non-null     str    
 11  TOT_EMP       87 non-null     float64
 12  EMP_PRSE      89 non-null     object 
 13  JOBS_1000     87 non-null     float64
 14  LOC_QUOTIENT  87 non-null     float64
 15  H_MEAN        88 non-null     float64
 16  A_MEAN        88 non-null     float64
 17  MEAN_PRSE     88 non-null     object 
 18  H_PCT10       88 non-null     object 
 1

## Essential cleaning checks

In [ ]:
# check for missing values
tech_df.isnull().sum()

AREA             0
AREA_TITLE       0
AREA_TYPE        0
PRIM_STATE       0
NAICS            0
NAICS_TITLE      0
I_GROUP          0
OWN_CODE         0
OCC_CODE         0
OCC_TITLE        0
O_GROUP          0
TOT_EMP          2
EMP_PRSE         0
JOBS_1000        2
LOC_QUOTIENT     2
H_MEAN           1
A_MEAN           1
MEAN_PRSE        1
H_PCT10          1
H_PCT25          1
H_MEDIAN         1
H_PCT75          1
H_PCT90          1
A_PCT10          1
A_PCT25          1
A_MEDIAN         1
A_PCT75          1
A_PCT90          1
ANNUAL          89
HOURLY          89
dtype: int64

In [25]:
tech_df.head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
91667,34900,"Napa, CA",4,CA,0,Cross-industry,cross-industry,1235,15-1211,Computer Systems Analysts,...,59.99,73.54,82.29,53860,101980,124790.0,152960,171160,NaN,NaN
91668,34900,"Napa, CA",4,CA,0,Cross-industry,cross-industry,1235,15-1231,Computer Network Support Specialists,...,38.55,41.93,56.49,50860,69680,80180.0,87210,117510,NaN,NaN
91669,34900,"Napa, CA",4,CA,0,Cross-industry,cross-industry,1235,15-1232,Computer User Support Specialists,...,38.01,49.83,60.92,54540,65110,79050.0,103650,126710,NaN,NaN
91670,34900,"Napa, CA",4,CA,0,Cross-industry,cross-industry,1235,15-1244,Network and Computer Systems Administrators,...,48.25,60.54,67.49,82340,89060,100360.0,125910,140380,NaN,NaN
91671,34900,"Napa, CA",4,CA,0,Cross-industry,cross-industry,1235,15-1252,Software Developers,...,73.62,85.78,108.55,98330,120430,153130.0,178420,225790,NaN,NaN


## Check and remove if exists duplicated rows

In [26]:
tech_df.duplicated().sum()

np.int64(0)

## Verify occupation uniqueness

In [ ]:
tech_df.groupby(["AREA_TITLE", "OCC_CODE"]).size().sort_values(ascending=False).head()

AREA_TITLE  OCC_CODE
Napa, CA    15-1211     1
            15-1231     1
            15-1232     1
            15-1244     1
            15-1252     1
dtype: int64

## Check for impossible values

In [39]:
#tech_df["TOT_EMP"].describe()
# tech_df["A_MEDIAN"].describe()
tech_df["A_MEAN"].describe()

count        88.000000
mean     133813.068182
std       35097.109596
min       78160.000000
25%      109252.500000
50%      123590.000000
75%      155347.500000
max      221710.000000
Name: A_MEAN, dtype: float64

## Verify Bay Area filter

In [41]:
tech_df["AREA_TITLE"].unique()

<StringArray>
[                          'Napa, CA',  'San Francisco-Oakland-Fremont, CA',
 'San Jose-Sunnyvale-Santa Clara, CA',         'Santa Cruz-Watsonville, CA',
            'Santa Rosa-Petaluma, CA',                        'Vallejo, CA']
Length: 6, dtype: str

In [ ]:
# check occupation titles unique count
#tech_df["OCC_TITLE"].nunique()
tech_df.describe()

20

## Save the bay area tech jobs data
- After filternig and cleaning

In [ ]:
tech_df.to_csv(
    "../data/processed/bay_area_tech_jobs.csv",
    index=False
)